In [ ]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import pandas as pd
import polars as pl
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from sklearn.cluster import DBSCAN, HDBSCAN

import sys

sys.path.append("..")

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter(dark_background=True)

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from src import SpotDetectionFunctions

SD_F = SpotDetectionFunctions.SpotDetection_Functions()

from src import SR_Functions

SRes_F = SR_Functions.SuperRes_Functions()

from src import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

import types

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [ ]:
# data_folder = '/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration/old/'
data_folder = "/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/Ximea_Calibration/"
# data_folder = r'C:\Users\jsb92\Cambridge University Dropbox\Joseph Beckwith\Chemistry\Lee\Data\Salix\Ximea_Calibration'
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()

In [ ]:
data_folders = np.array(
    [
        r"/media/jbeckwith/Ezra Seagat/JSB/20250523_HeLa_STORM/data/Cell4/",
        r"/media/jbeckwith/Ezra Seagat/JSB/20250523_HeLa_STORM/data/Cell3/",
    ]
)

In [ ]:
example_folder = data_folders[0]

In [ ]:
localisation_files = H_F.file_search(example_folder, ".h5", "")

In [ ]:
columns = [
    "xc",
    "yc",
    "s_x",
    "s_y",
    "bg_B",
    "bg_G",
    "bg_R",
    "A_B",
    "A_G",
    "A_R",
    "chi_sqr",
    "frame",
    "xc_err",
    "yc_err",
    "s_x_err",
    "s_y_err",
    "bg_B_err",
    "bg_G_err",
    "bg_R_err",
    "A_B_err",
    "A_G_err",
    "A_R_err",
]


loc_data = pd.read_hdf(localisation_files[0])
loc_data = loc_data[loc_data["xc_err"] < 1]
loc_data = loc_data[loc_data["yc_err"] < 1]

loc_data = loc_data[loc_data["xc_err"] > 0]
loc_data = loc_data[loc_data["yc_err"] > 0]

loc_data = loc_data[loc_data["A_B"] + loc_data["A_G"] + loc_data["A_R"] < 10000]
loc_data = loc_data[loc_data["A_B"] + loc_data["A_G"] + loc_data["A_R"] > 500]
loc_data = loc_data[loc_data["A_B"] > 0]
loc_data = loc_data[loc_data["A_G"] > 0]
loc_data = loc_data[loc_data["A_R"] > 0]

loc_data = loc_data[loc_data["chi_sqr"] < 2]

In [ ]:
X = np.vstack([loc_data["xc"], loc_data["yc"]]).T
loc_precision = 0.5 * (np.mean(loc_data["xc_err"]) + np.mean(loc_data["yc_err"]))
hdb = DBSCAN(eps=loc_precision, min_samples=2000)
hdb.fit(X)

In [ ]:
fiducial_1 = loc_data[loc_data["xc"] > 100]
fiducial_1 = fiducial_1[fiducial_1["xc"] < 104]
fiducial_1 = fiducial_1[fiducial_1["yc"] < 404]
fiducial_1 = fiducial_1[fiducial_1["yc"] > 398]
fiducial_1["xc"] = fiducial_1["xc"] * 69 - np.mean(
    fiducial_1["xc"].to_numpy()[0:200] * 69
)
fiducial_1["yc"] = fiducial_1["yc"] * 69 - np.mean(
    fiducial_1["yc"].to_numpy()[0:200] * 69
)

In [ ]:
fiducial_2 = loc_data[loc_data["xc"] > 520]
fiducial_2 = fiducial_2[fiducial_2["xc"] < 540]
fiducial_2 = fiducial_2[fiducial_2["yc"] < 160]
fiducial_2 = fiducial_2[fiducial_2["yc"] > 140]
fiducial_2["xc"] = fiducial_2["xc"] * 69 - np.mean(
    fiducial_2["xc"].to_numpy()[0:200] * 69
)
fiducial_2["yc"] = fiducial_2["yc"] * 69 - np.mean(
    fiducial_2["yc"].to_numpy()[0:200] * 69
)

In [ ]:
fiducial_3 = loc_data[loc_data["xc"] > 600]
fiducial_3 = fiducial_3[fiducial_3["xc"] < 620]
fiducial_3 = fiducial_3[fiducial_3["yc"] < 200]
fiducial_3 = fiducial_3[fiducial_3["yc"] > 180]
fiducial_3["xc"] = fiducial_3["xc"] * 69 - np.mean(
    fiducial_3["xc"].to_numpy()[0:200] * 69
)
fiducial_3["yc"] = fiducial_3["yc"] * 69 - np.mean(
    fiducial_3["yc"].to_numpy()[0:200] * 69
)

In [ ]:
fiducial_4 = loc_data[loc_data["xc"] > 730]
fiducial_4 = fiducial_4[fiducial_4["xc"] < 740]
fiducial_4 = fiducial_4[fiducial_4["yc"] < 200]
fiducial_4 = fiducial_4[fiducial_4["yc"] > 192]
fiducial_4["xc"] = fiducial_4["xc"] * 69 - np.mean(
    fiducial_4["xc"].to_numpy()[0:200] * 69
)
fiducial_4["yc"] = fiducial_4["yc"] * 69 - np.mean(
    fiducial_4["yc"].to_numpy()[0:200] * 69
)

In [ ]:
smoother_x = np.full([4, int(np.max(loc_data["frame"]))], np.NAN)
smoother_y = np.full([4, int(np.max(loc_data["frame"]))], np.NAN)

In [ ]:
for i in np.arange(int(np.max(loc_data["frame"]))):
    if np.isin(i, fiducial_1["frame"]):
        loc = np.argmin(np.abs(fiducial_1["frame"].to_numpy() - i))
        smoother_x[0, i] = fiducial_1["xc"].to_numpy()[loc]
        smoother_y[0, i] = fiducial_1["yc"].to_numpy()[loc]
    if np.isin(i, fiducial_2["frame"]):
        loc = np.argmin(np.abs(fiducial_2["frame"].to_numpy() - i))
        smoother_x[1, i] = fiducial_2["xc"].to_numpy()[loc]
        smoother_y[1, i] = fiducial_2["yc"].to_numpy()[loc]
    if np.isin(i, fiducial_3["frame"]):
        loc = np.argmin(np.abs(fiducial_3["frame"].to_numpy() - i))
        smoother_x[2, i] = fiducial_3["xc"].to_numpy()[loc]
        smoother_y[2, i] = fiducial_3["yc"].to_numpy()[loc]
    if np.isin(i, fiducial_4["frame"]):
        loc = np.argmin(np.abs(fiducial_4["frame"].to_numpy() - i))
        smoother_x[3, i] = fiducial_4["xc"].to_numpy()[loc]
        smoother_y[3, i] = fiducial_4["yc"].to_numpy()[loc]

In [ ]:
smoother_x_avg = np.nanmean(smoother_x, axis=0)
smoother_y_avg = np.nanmean(smoother_y, axis=0)

In [ ]:
def moving_average(a, n=3):
    ret = np.nancumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1 :] / n

In [ ]:
loc_data_copy = deepcopy(loc_data)

In [ ]:
photons = loc_data_copy["A_B"] + loc_data_copy["A_G"] + loc_data_copy["A_R"]
loc_data_copy = loc_data_copy[loc_data_copy["A_B"] / photons > 1e-3]
photons = loc_data_copy["A_B"] + loc_data_copy["A_G"] + loc_data_copy["A_R"]
loc_data_copy = loc_data_copy[loc_data_copy["A_G"] / photons > 1e-3]
photons = loc_data_copy["A_B"] + loc_data_copy["A_G"] + loc_data_copy["A_R"]
loc_data_copy = loc_data_copy[loc_data_copy["A_R"] / photons > 1e-3]
loc_data_copy = loc_data_copy[loc_data_copy["s_x"] * 69 > 50]
loc_data_copy = loc_data_copy[loc_data_copy["s_y"] * 69 > 50]
loc_data_copy = loc_data_copy[loc_data_copy["A_B_err"] < 300]
loc_data_copy = loc_data_copy[loc_data_copy["A_G_err"] < 300]
loc_data_copy = loc_data_copy[loc_data_copy["A_R_err"] < 300]

In [ ]:
photons = loc_data_copy["A_B"] + loc_data_copy["A_G"] + loc_data_copy["A_R"]
plt.hist(photons, 1000)
# plt.xlim([0, 100])
plt.show()

In [ ]:
for i in np.arange(int(np.max(loc_data["frame"]))):
    loc_data_copy[loc_data_copy["frame"] == i]["xc"] = (
        loc_data_copy[loc_data_copy["frame"] == i]["xc"] - smoother_x_avg[i] / 69
    )
    loc_data_copy[loc_data_copy["frame"] == i]["yc"] = (
        loc_data_copy[loc_data_copy["frame"] == i]["yc"] - smoother_y_avg[i] / 69
    )

In [ ]:
test = loc_data_copy[loc_data_copy["s_y_err"] * 69 < 40]
test = test[test["s_x_err"] * 69 < 40]
test = test[test["A_B"] + test["A_G"] + test["A_R"] < 5000]
test = test[test["A_B"] + test["A_G"] + test["A_R"] > 1000]
plt.scatter(test["xc"], test["yc"], alpha=0.05, s=0.01)
plt.xlim([0, 820])
plt.ylim([0, 1250])
plt.show()

In [ ]:
X = np.vstack([test["xc"], test["yc"]]).T
loc_precision = 0.5 * (np.mean(test["xc_err"]) + np.mean(test["yc_err"]))
hdb = DBSCAN(eps=loc_precision, min_samples=10)
hdb.fit(X)

In [ ]:
test2 = test[hdb.labels_ < 0]

In [ ]:
from src import render

locs, rendered_object = render.render(
    test2.to_records(index=False),
    viewport=((0, 0), (1250, 815)),
    blur_method="gaussian",
    oversampling=2,
)

In [ ]:
fig, axs = plotter.two_column_plot()

axs = plotter.image_plot(
    axs=axs,
    data=rendered_object.T,
    vmin=np.percentile(rendered_object, 1),
    vmax=np.percentile(rendered_object, 99.9),
    cbar="on",
    cbarlabel="localisations",
    pixelsize=69 / 2,
)
import matplotlib.patches as patches

rect = patches.Rectangle(
    (900, 800), 300, 300, linewidth=0.5, edgecolor="white", facecolor="none"
)

# Add the patch to the Axes
axs.add_patch(rect)

plt.savefig("Exemplar_SuperRes_box.svg", dpi=1200, format="svg")

In [ ]:
fig, axs = plotter.one_column_plot()

axs = plotter.image_plot(
    axs=axs,
    data=rendered_object.T[800:1100, 900:1200],
    vmin=np.percentile(rendered_object, 1),
    vmax=np.percentile(rendered_object, 99.9),
    cbar="off",
    cbarlabel="localisations",
    pixelsize=69 / 2,
)


plt.savefig("Exemplar_SuperRes_Zoomin.svg", dpi=1200, format="svg")

In [ ]:
locs, rendered_object_nonSR = render.render(
    test2.to_records(index=False),
    viewport=((0, 0), (1250, 815)),
    blur_method="gaussian",
    min_blur_width=3.3,
    oversampling=2,
)

In [ ]:
fig, axs = plotter.two_column_plot()

axs = plotter.image_plot(
    axs=axs,
    data=rendered_object_nonSR.T,
    vmin=np.percentile(rendered_object, 1),
    vmax=np.percentile(rendered_object, 99.9),
    cbar="on",
    cbarlabel="localisations",
    pixelsize=69 / 2,
)

rect = patches.Rectangle(
    (900, 800), 300, 300, linewidth=0.5, edgecolor="white", facecolor="none"
)

# Add the patch to the Axes
axs.add_patch(rect)

plt.savefig("Exemplar_NonSuperRes_Box.svg", dpi=1200, format="svg")

In [ ]:
fig, axs = plotter.one_column_plot()

axs = plotter.image_plot(
    axs=axs,
    data=rendered_object_nonSR.T[800:1100, 900:1200],
    vmin=np.percentile(rendered_object, 1),
    vmax=np.percentile(rendered_object, 99.9),
    cbar="on",
    cbarlabel="localisations",
    pixelsize=69 / 2,
)


plt.savefig("Exemplar_NonSuperRes_Zoomin.svg", dpi=1200, format="svg")

In [ ]:
def average_parameters(data, dbscan_labels):
    labels = np.sort(np.unique(dbscan_labels))
    labels = labels[labels > -1]
    dict_obj = {}
    dict_obj["photons"] = np.zeros(len(labels))
    dict_obj["frames"] = np.zeros(len(labels))
    for column in np.array(data.columns):
        if column == "index":
            continue
        else:
            dict_obj[column] = np.zeros(len(labels))

    for label in labels:
        for column in np.array(data.columns):
            if column == "index":
                continue
            elif column in ["A_B", "A_G", "A_R"]:
                dict_obj[column][label] = np.sum(data[column][dbscan_labels == label])
            else:
                dict_obj[column][label] = np.mean(data[column][dbscan_labels == label])
        dict_obj["frames"][label] = len(data[column][dbscan_labels == label])
        dict_obj["photons"][label] = (
            np.sum(data["A_B"][dbscan_labels == label])
            + np.sum(data["A_G"][dbscan_labels == label])
            + np.sum(data["A_R"][dbscan_labels == label])
        )
    dict_obj["A_B"] = dict_obj["A_B"] / dict_obj["photons"]
    dict_obj["A_G"] = dict_obj["A_G"] / dict_obj["photons"]
    dict_obj["A_R"] = dict_obj["A_R"] / dict_obj["photons"]
    df = pd.DataFrame.from_dict(dict_obj)
    return df

In [ ]:
def collect_traces(data, dbscan_labels, image_stack, image_size=12):
    labels = np.sort(np.unique(dbscan_labels))
    labels = labels[labels > -1]
    trace_matrix = np.zeros([len(labels), image_stack.shape[0]])
    locations = np.zeros([2, len(labels)])

    for i, label in enumerate(labels):
        locations[0, i] = np.nanmean(data["xc"][dbscan_labels == label].to_numpy())
        locations[1, i] = np.nanmean(data["yc"][dbscan_labels == label].to_numpy())
        xmin = int(locations[0, i]) - int(image_size / 2)
        xmax = int(locations[0, i]) + int(image_size / 2)
        ymin = int(locations[1, i]) - int(image_size / 2)
        ymax = int(locations[1, i]) + int(image_size / 2)
        trace_matrix[i, :] = np.sum(
            np.sum(image_stack[:, xmin:xmax, ymin:ymax], axis=-1), axis=-1
        )
        print(
            "Summed trace {}/{}".format(i + 1, len(labels)),
            end="\r",
            flush=True,
        )

    return locations, trace_matrix

In [ ]:
def extract_single_molecules(localisation_files, chi_val=None):
    columns = [
        "xc",
        "yc",
        "s_x",
        "s_y",
        "bg_B",
        "bg_G",
        "bg_R",
        "A_B",
        "A_G",
        "A_R",
        "chi_sqr",
        "frame",
        "xc_err",
        "yc_err",
        "s_x_err",
        "s_y_err",
        "bg_B_err",
        "bg_G_err",
        "bg_R_err",
        "A_B_err",
        "A_G_err",
        "A_R_err",
    ]

    for i, file in enumerate(localisation_files):
        loc_data = pd.read_hdf(file, columns=columns)
        if chi_val is None:
            chi_val = np.median(loc_data["chi_sqr"])
        loc_data = loc_data[loc_data["chi_sqr"] < chi_val]
        loc_data = loc_data[loc_data["xc_err"] < 1]
        loc_data = loc_data[loc_data["yc_err"] < 1]
        loc_data = loc_data[loc_data["A_B"] + loc_data["A_G"] + loc_data["A_R"] < 10000]
        loc_data = loc_data[loc_data["A_B"] + loc_data["A_G"] + loc_data["A_R"] > 500]
        loc_data = loc_data.reset_index()
        X = np.vstack([loc_data["xc"], loc_data["yc"]]).T
        loc_precision = 0.5 * (
            np.mean(loc_data["xc_err"]) + np.mean(loc_data["yc_err"])
        )
        hdb = HDBSCAN(min_cluster_size=10, cluster_selection_epsilon=loc_precision)
        hdb.fit(X)
        df = average_parameters(loc_data, hdb.labels_)
        photons = loc_data["A_B"] + loc_data["A_G"] + loc_data["A_R"]
        loc_data["photons"] = photons
        for column in np.array(loc_data.columns):
            if column in ["A_B", "A_G", "A_R"]:
                loc_data[column] = loc_data[column] / photons
        if i == 0:
            single_frame_database = loc_data
            single_molecule_database = df
        else:
            single_frame_database = pd.concat([single_frame_database, loc_data])
            single_molecule_database = pd.concat([single_molecule_database, df])
    return single_molecule_database.reset_index(), single_frame_database.reset_index()

In [ ]:
def extract_single_molecule_traces(localisation_file, chi_val=None):
    columns = [
        "xc",
        "yc",
        "s_x",
        "s_y",
        "bg_B",
        "bg_G",
        "bg_R",
        "A_B",
        "A_G",
        "A_R",
        "chi_sqr",
        "frame",
        "xc_err",
        "yc_err",
        "s_x_err",
        "s_y_err",
        "bg_B_err",
        "bg_G_err",
        "bg_R_err",
        "A_B_err",
        "A_G_err",
        "A_R_err",
    ]

    image_file = localisation_file.split(".")[0] + ".ome.tif"
    metadata = localisation_file.split(".")[0] + "_metadata.txt"
    x_coord, y_coord, width, height = IO.metadata_reader_imageJ(metadata)
    raw_data, image_data, _, _ = IO.read_tiff_tophotoelectrons(        image_file,
        smoothing_function,
        gain_map=gain[x_coord : x_coord + width, y_coord : y_coord + height],
        offset_map=offset[x_coord : x_coord + width, y_coord : y_coord + height],
        rqe=rqe[x_coord : x_coord + width, y_coord : y_coord + height],
        read_noise=readnoise[x_coord : x_coord + width, y_coord : y_coord + height],
        frame=np.arange(750),
    )
    loc_data = pd.read_hdf(localisation_file, columns=columns)
    if chi_val is None:
        chi_val = np.median(loc_data["chi_sqr"])
    loc_data = loc_data[loc_data["chi_sqr"] < chi_val]
    loc_data = loc_data[loc_data["xc_err"] < 1]
    loc_data = loc_data[loc_data["yc_err"] < 1]
    loc_data = loc_data[loc_data["A_B"] + loc_data["A_G"] + loc_data["A_R"] < 10000]
    loc_data = loc_data[loc_data["A_B"] + loc_data["A_G"] + loc_data["A_R"] > 500]
    loc_data = loc_data.reset_index()
    X = np.vstack([loc_data["xc"], loc_data["yc"]]).T
    loc_precision = 0.5 * (np.mean(loc_data["xc_err"]) + np.mean(loc_data["yc_err"]))
    hdb = DBSCAN(min_samples=10, eps=loc_precision)
    hdb.fit(X)
    locations, trace_matrix = collect_traces(loc_data, hdb.labels_, image_data)
    return locations, trace_matrix, image_data

In [ ]:
locations, trace_matrix, image_data = extract_single_molecule_traces(
    localisation_files[0], 2
)

In [ ]:
# good traces
# 80
# 123
# 47

In [ ]:
locations_forplot = locations[:, [80, 47, 123]]
trace_matrix_forplot = trace_matrix[[80, 47, 123], :]
trace_matrix_forplot = trace_matrix_forplot.T - np.min(trace_matrix_forplot, axis=1)
trace_matrix_forplot = trace_matrix_forplot.T

In [ ]:
fig, axs = plotter.two_column_plot(
    ncolumns=3, nrows=2, widthratio=[1, 1, 1], heightratio=[1, 1], height=6, width=9
)

folder = "/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Talks+Posters/Conference Presentations/20250627_RSCPhotochem/fig/multicolour_camera/dyes/"
plotter.make_animated_gif_bleaching(
    fig,
    axs,
    locations_forplot,
    trace_matrix_forplot,
    image_data[:450, :, :],
    14,
    np.linspace(0, 749 * 0.1, 750)[:450],
    os.path.join(folder, "LD655_bleaching.gif"),
)

In [ ]:
plt.plot(trace_matrix[80, :])
plt.xlim([0, 60])

In [ ]:
single_molecule_database, single_frame_database = extract_single_molecules(
    localisation_files
)

In [ ]:
single_frame_database.to_hdf(
    os.path.join(example_folder, "Single_frame_database.h5"), key="data", format="table"
)

In [ ]:
single_molecule_database.to_hdf(
    os.path.join(example_folder, "Single_molecule_database.h5"),
    key="data",
    format="table",
)

In [ ]:
notch_filter = "semrock-nf03-405-488-561-635e"
dichroic_mirror = "semrock-di03-r405-488-561-635-t1-25x36"
shortpass_filter = "semrock-bsp01-785r"
longpass_red_filter = "semrock-blp01-635r"

longpass_blue_filter = "semrock-blp01-488r"
bandpass_488_filter = "semrock-ff01-520-44"
filters = [longpass_red_filter, dichroic_mirror]
dye = "LD 655"
average_emission_wavelengths, dye_pixel_efficiency = (
    S_F.get_pixel_fractions_dye_and_filters(dye, filters, wavelength, pixel_QYs)
)
dye_pixel_efficiency = dye_pixel_efficiency / np.sum(dye_pixel_efficiency)

In [ ]:
LD655 = S_F.get_dye_or_filter_data(dye, wavelength)
filter = np.prod(
    S_F.get_dye_or_filter_data(filters, wavelength, dye_or_filter=False), axis=0
)

In [ ]:
filter = filter * objective_T

In [ ]:
average_emission_wavelengths, dye_pixel_efficiency = S_F.get_pixel_fractions_rawspectra(
    LD655 * filter, wavelength, pixel_QYs
)

In [ ]:
filters = S_F.get_dye_or_filter_data(filters, wavelength, dye_or_filter=False)

In [ ]:
fig, axs = plotter.one_column_plot(npanels=2, ratios=[1, 1], height=5)

ax2 = axs[0].twinx()  # instantiate a second Axes that shares the same x-axis
ax2.set_ylabel(
    "transmission/%", color="#ffc0cb"
)  # we already handled the x-label with ax1

linestyles = np.array(["--", "-.", ":"])
for i in np.arange(len(filters)):
    if i != 2:
        ax2.plot(
            wavelength, 100 * filters[i, :], color="#ffc0cb", lw=0.5, ls=linestyles[i]
        )
    if i == 2:
        ax2.plot(
            wavelength,
            100 * filters[i, :],
            color="#ffc0cb",
            lw=0.5,
            ls=linestyles[i],
            label="microscope filters",
        )

ax2.set_ylim([0, 110])
ax2.legend(loc="best")


axs[0] = plotter.line_plot(
    axs=axs[0],
    x=wavelength,
    y=np.squeeze(LD655) / np.max(LD655),
    color="white",
    xaxislabel="wavelength/nm",
    yaxislabel="fluo/norm",
    label="LD 655",
)

axs[0].set_ylim(0, 1.1)
axs[0].set_xlim(550, 800)
axs[0].legend(loc="best")

axs[1] = plotter.line_plot(
    axs=axs[1],
    x=wavelength,
    y=np.squeeze(filtered_L655) / np.max(filtered_L655),
    color="white",
    xaxislabel="wavelength/nm",
    yaxislabel="fluo/norm",
    label="LD 655 after filters",
)
axs[1].legend(loc="best")

axs[1].set_ylim(0, 1.05)
axs[1].set_xlim(550, 800)

folder = "/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Talks+Posters/Conference Presentations/20250627_RSCPhotochem/fig/multicolour_camera/dyes/"
plt.savefig(os.path.join(folder, "LD655_Spectra.svg"), dpi=600)
plt.show()

In [ ]:
plt.plot(wavelength, filter.T, color="white")

In [ ]:
cf500_maxt = 0.25
cf500_maxl = 0.9
cf500_maxr = 0.35
cf500_ts = 0.25

bodipy_maxl = 1
bodipy_maxt = 0.45
bodipy_maxr = 0.45
bodipy_ts = 0.45

ld655_maxt = 0.85
ld655_maxl = 0.35
ld655_maxr = 0.2
ld655_ts = 0.2

In [ ]:
fig, ax = plotter.one_column_plot(npanels=2, ratios=[0.5, 1])


fig, ax = plotter.ternary_contour_plot(
    fig,
    ax,
    single_molecule_database["A_R"],
    single_molecule_database["A_G"],
    single_molecule_database["A_B"],
    gridsize=300,
    R=dye_pixel_efficiency[2],
    G=dye_pixel_efficiency[1],
    B=dye_pixel_efficiency[0],
    maxt=ld655_maxt,
    maxl=ld655_maxl,
    maxr=ld655_maxr,
    trianglesize=ld655_ts,
    maj_loc=0.05,
    min_loc=0.025,
    lws=1,
    ecolour="red",
)

n_molecules = len(single_molecule_database["photons"])
ax[0] = plotter.histogram_plot(
    axs=ax[0],
    data=single_molecule_database["photons"],
    bins=np.histogram_bin_edges(single_molecule_database["photons"], "fd"),
    histcolor="#d3d3d3",
    xaxislabel="photons per molecule",
    label=r"N$_\mathsf{molecules}$=" + str(int(n_molecules)),
)
ax[0].set_xlim([0, 2e6])
ax[0].legend(loc="best", frameon=False)

folder = "/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Talks+Posters/Conference Presentations/20250627_RSCPhotochem/fig/multicolour_camera/dyes/"
plt.savefig(os.path.join(folder, "LD655_permolecule_objective.svg"), dpi=600)
plt.show()

In [ ]:
fig, ax = plotter.one_column_plot(npanels=2, ratios=[0.5, 1])


fig, ax = plotter.ternary_contour_plot(
    fig,
    ax,
    single_frame_database["A_R"],
    single_frame_database["A_G"],
    single_frame_database["A_B"],
    gridsize=100,
    R=dye_pixel_efficiency[2],
    G=dye_pixel_efficiency[1],
    B=dye_pixel_efficiency[0],
    maxt=bodipy_maxt,
    maxl=bodipy_maxl,
    maxr=bodipy_maxr,
    trianglesize=bodipy_ts,
    maj_loc=0.2,
    min_loc=0.1,
    lws=1,
    ecolour="red",
)

n_molecules = len(single_frame_database["photons"])
ax[0] = plotter.histogram_plot(
    axs=ax[0],
    data=single_frame_database["photons"],
    bins=np.histogram_bin_edges(single_frame_database["photons"], "fd"),
    histcolor="#d3d3d3",
    xaxislabel="photons per frame",
    label=r"N$_\mathsf{molecules}$=" + str(int(n_molecules)),
)

ax[0].legend(loc="best", frameon=False)

folder = "/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Talks+Posters/Conference Presentations/20250627_RSCPhotochem/fig/multicolour_camera/dyes/"
# plt.savefig(os.path.join(folder, 'BODIPY_PVA_perframe.svg'), dpi=600)
plt.show()